# Chapter 32: Project: An LRU Cache

The programs from Chapter 32 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## The Code

`lru_cache.py`

In [1]:
"""An LRU (least recently used) cache with a fixed capacity.

A dict finds an entry's node in constant time; a doubly linked list
keeps the entries in order of use, most recent at the front. Using an
entry moves its node to the front; when the cache is full, the node at
the back (the least recently used) is evicted. Every operation is O(1).
"""


class _Node:
    __slots__ = ("key", "value", "prev", "next")

    def __init__(self, key=None, value=None):
        self.key, self.value = key, value
        self.prev = self.next = self


class LRUCache:
    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError("capacity must be at least 1")
        self.capacity = capacity
        self._map = {}  # key -> node
        self._sentinel = _Node()  # front: sentinel.next; back: .prev
        self.hits = self.misses = 0

    def __len__(self):
        return len(self._map)

    def _unlink(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def _push_front(self, node):
        first = self._sentinel.next
        node.prev, node.next = self._sentinel, first
        first.prev = node
        self._sentinel.next = node

    def get(self, key, default=None):
        node = self._map.get(key)
        if node is None:
            self.misses += 1
            return default
        self.hits += 1
        self._unlink(node)  # used just now: move to the front
        self._push_front(node)
        return node.value

    def put(self, key, value):
        node = self._map.get(key)
        if node:
            node.value = value
            self._unlink(node)
        else:
            if len(self._map) == self.capacity:
                oldest = self._sentinel.prev  # the back of the list
                self._unlink(oldest)
                del self._map[oldest.key]
            node = _Node(key, value)
            self._map[key] = node
        self._push_front(node)

    def keys(self):
        """Keys from most to least recently used."""
        node = self._sentinel.next
        while node is not self._sentinel:
            yield node.key
            node = node.next

## Watching It Work

`lru_demo.py`

In [2]:
"""An LRU cache of capacity 3, step by step."""

from lru_cache import LRUCache

c = LRUCache(3)
for step in [
    "put a",
    "put b",
    "put c",
    "get a",
    "put d",
    "get b",
    "get c",
    "put e",
]:
    action, key = step.split()
    if action == "put":
        c.put(key, key.upper())
        result = ""
    else:
        result = f"-> {c.get(key)}"
    print(f"{step} {result:<7} cache (newest first): {list(c.keys())}")
print(f"hits {c.hits}, misses {c.misses}")

put a         cache (newest first): ['a']
put b         cache (newest first): ['b', 'a']
put c         cache (newest first): ['c', 'b', 'a']
get a -> A    cache (newest first): ['a', 'c', 'b']
put d         cache (newest first): ['d', 'a', 'c']
get b -> None cache (newest first): ['d', 'a', 'c']
get c -> C    cache (newest first): ['c', 'd', 'a']
put e         cache (newest first): ['e', 'c', 'd']
hits 2, misses 1


## A Memoizing Decorator

`memoize.py`

In [3]:
"""A memoizing decorator with a bounded cache, like the standard
library's functools.lru_cache, built on our LRUCache."""

import functools

from lru_cache import LRUCache

MISSING = object()


def lru_memoize(capacity):
    def decorate(func):
        cache = LRUCache(capacity)

        @functools.wraps(func)
        def wrapper(*args):
            result = cache.get(args, MISSING)
            if result is MISSING:
                result = func(*args)
                cache.put(args, result)
            return result

        wrapper.cache = cache
        return wrapper

    return decorate


@lru_memoize(capacity=100)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


@functools.lru_cache(maxsize=100)
def fib_std(n):
    return n if n < 2 else fib_std(n - 1) + fib_std(n - 2)


print(fib(80), fib_std(80))
print("ours:", fib.cache.hits, "hits,", fib.cache.misses, "misses")
print("functools:", fib_std.cache_info())

23416728348467685 23416728348467685
ours: 78 hits, 81 misses
functools: CacheInfo(hits=78, misses=81, maxsize=100, currsize=81)


## How Big Should the Cache Be?

`hit_rate.py`

In [4]:
"""How big should a cache be? Requests for 10,000 different pages,
where a few popular pages get most of the traffic (page k is requested
in proportion to 1 / k). Hit rate for different capacities."""

import random

from lru_cache import LRUCache

rng = random.Random(2026)
pages = range(1, 10_001)
weights = [1 / k for k in pages]
requests = rng.choices(pages, weights, k=200_000)

print(f"{'capacity':>9} {'hit rate':>9}")
for capacity in [10, 100, 1_000, 5_000]:
    cache = LRUCache(capacity)
    for page in requests:
        if cache.get(page) is None:
            cache.put(page, True)  # a miss: fetch it, then keep it
    print(f"{capacity:>9,} {cache.hits / len(requests):>9.1%}")

 capacity  hit rate
       10     13.2%


      100     39.3%


    1,000     67.6%
    5,000     88.8%


## Timing

`bench_lru.py`

In [5]:
"""A mixed workload of gets and puts: our LRUCache, an OrderedDict
version, and a dict with no limit (the fastest possible, unbounded)."""

import random
from collections import OrderedDict

import bench
from lru_cache import LRUCache

rng = random.Random(2026)
keys = [rng.randint(0, 2_000) for _ in range(100_000)]


class OrderedLRU:
    def __init__(self, capacity):
        self.capacity, self.d = capacity, OrderedDict()

    def get(self, key, default=None):
        if key in self.d:
            self.d.move_to_end(key)
            return self.d[key]
        return default

    def put(self, key, value):
        self.d[key] = value
        self.d.move_to_end(key)
        if len(self.d) > self.capacity:
            self.d.popitem(last=False)


class Unbounded(dict):
    """A dict with the same get() and put() methods; it never evicts."""

    put = dict.__setitem__


def workload(cache):
    for k in keys:
        if cache.get(k) is None:
            cache.put(k, k)


caches = {
    "LRUCache": lambda: LRUCache(500),
    "OrderedDict LRU": lambda: OrderedLRU(500),
    "plain dict": Unbounded,
}
for name, make in caches.items():
    cache = make()
    seconds = bench.measure(lambda c=cache: workload(c))
    print(f"{name:>16}: {bench.show_time(seconds)} for 100,000 requests")

        LRUCache: 37.6 ms for 100,000 requests
 OrderedDict LRU: 21.7 ms for 100,000 requests


      plain dict: 3.96 ms for 100,000 requests
